#  Intro to AI workflows using LangChain and Small LM
## Using LangChain with llama-cpp-python

This notebook introduces **LangChain**, a powerful framework for building applications with Language Models. It builds on concepts from the [LlamaCpp_SmallLM_Demo.ipynb](LlamaCpp_SmallLM_Demo.ipynb) notebook.

### What is LangChain?

[LangChain](https://python.langchain.com/) is a framework for developing applications powered by language models. It provides tools and abstractions that make it easier to:
- Build complex prompts with templates
- Chain multiple operations together
- Add memory to conversations
- Integrate with various data sources and tools

### Key Features of LangChain:

1. **Prompt Templates**: Reusable, parameterized prompts
2. **Chains**: Combine multiple components into workflows
3. **Memory**: Keep track of conversation history
4. **Model Agnostic**: Works with OpenAI, local models, and more
5. **Rich Ecosystem**: Integrations with vector databases, tools, and APIs

### Why use LangChain?

| Feature | Without LangChain | With LangChain |
|---------|------------------|----------------|
| Prompt Management | Manual string formatting | Reusable templates with variables |
| Conversation Memory | Manual list management | Built-in memory classes |
| Complex Workflows | Custom code for each step | Pre-built chains and components |
| Code Reusability | Copy-paste prompts | Modular, composable components |

**Fun Fact**: LangChain works seamlessly with llama-cpp-python, so you can use the same local models we've been using!

### Resources

- LangChain [Documentation](https://python.langchain.com/)
- LangChain [GitHub Repository](https://github.com/langchain-ai/langchain)
- LangChain [Community](https://github.com/langchain-ai/langchain/discussions)

### Attribution

Notebook developed based on the teaching approach established by Greg Merritt and Eric Van Dusen in the SmallLM series.

## 1. Environment Setup

### Installing LangChain and Dependencies

We'll need to install:
1. **langchain**: The core LangChain library
2. **langchain-community**: Community integrations (includes llama-cpp support)
3. **llama-cpp-python**: For running local models (if not already installed)

**Note**: The first time you run this, it may take a minute to install all dependencies.

In [ ]:
# Install LangChain and dependencies
try:
    from langchain.prompts import PromptTemplate
    from langchain_community.llms import LlamaCpp
except ImportError:
    %pip install langchain langchain-community llama-cpp-python
    from langchain.prompts import PromptTemplate
    from langchain_community.llms import LlamaCpp

### Understanding the LangChain Components

We'll use three main components in this notebook:

| Component | Purpose | Example Use |
|-----------|---------|-------------|
| `PromptTemplate` | Create reusable prompt templates | "Explain {topic} in simple terms" |
| `LlamaCpp` | Interface to local GGUF models | Load and run qwen model |
| `ConversationBufferMemory` | Remember conversation history | Build chatbots with context |
| `LLMChain` | Chain prompts with models | Combine template + model + memory |

### Let's check out our local filesystem path and whether we have files downloaded

We need to locate where our `.gguf` model files are stored. This is the same setup as in the LlamaCpp notebook.

### Approach 1 - if a Shared Hub is being used 

In [ ]:
# This only worked for FA 25 workshop on Cal ICOR Hub
#!ls /home/jovyan/shared_readwrite

In [ ]:
# On Cal-ICOR workshop hub (JupyterCon Nov 2025)
!ls /home/jovyan/shared/

### Approach 2 - if a local machine is being used

In [ ]:
#This is my local path to a directory called shared-rw
!ls shared-rw

In [ ]:
# or the full path ( this is on my laptop) 
!ls /Users/ericvandusen/Documents/GitHub/SmallLM-SP25/shared-rw

### 1.1 Pick your environment - Local vs Hub - and set the Path

In [ ]:
# set the model path parameter depending on where you are computing
model_directory = "/home/jovyan/shared/"

In [ ]:
# set the model path parameter depending on where you are computing
#model_directory = "/Users/ericvandusen/Documents/GitHub/SmallLM-SP25/shared-rw"

### 1.2 Loading the Model with LangChain

LangChain provides a `LlamaCpp` wrapper that makes it easy to use local GGUF models. The wrapper handles all the complexity of interfacing with llama-cpp-python.

**Key parameters:**
- `model_path`: Full path to the .gguf file
- `temperature`: Controls randomness (0 = deterministic, higher = more random)
- `max_tokens`: Maximum length of generated response
- `n_ctx`: Context window size
- `verbose`: Print loading information

**This may take a few seconds to load the model.**

In [ ]:
import os

# Define the model filename
model_name = "qwen2-1_5b-instruct-q4_0.gguf"

# Create the full path to the model
model_path = os.path.join(model_directory, model_name)

# Load the model using LangChain's LlamaCpp wrapper
llm = LlamaCpp(
    model_path=model_path,
    temperature=0.75,
    max_tokens=200,
    n_ctx=2048,
    verbose=True,
)

print(f"\n✓ Model loaded successfully with LangChain: {model_name}")

## 2. Basic Usage: Calling the Model Directly

Before we use LangChain's advanced features, let's see how to make a simple call to the model. This is similar to what we did in the LlamaCpp notebook, but now using LangChain's interface.

**Note**: The model may take a moment to generate a response.

In [ ]:
# Simple call to the model
question = "What is artificial intelligence?"

response = llm.invoke(question)

print("Question:", question)
print("\nResponse:")
print(response)

## 3. Prompt Templates: Making Reusable Prompts

One of LangChain's most useful features is **Prompt Templates**. Instead of manually formatting strings every time, you create a template once and reuse it with different variables.

### Why Use Prompt Templates?

**Without templates:**
```python
prompt1 = f"Explain {topic1} in simple terms"
prompt2 = f"Explain {topic2} in simple terms"
# ... error-prone, repetitive
```

**With templates:**
```python
template = PromptTemplate(
    template="Explain {topic} in simple terms",
    input_variables=["topic"]
)
prompt1 = template.format(topic="quantum computing")
prompt2 = template.format(topic="blockchain")
```

### 3a. Creating a Simple Prompt Template

Let's create a template for explaining topics to college freshmen.

In [ ]:
from langchain.prompts import PromptTemplate

# Create a prompt template
template = """
You are a helpful tutor explaining concepts to college freshmen.
Explain {topic} in simple, clear terms that a first-year student can understand.
Use examples when helpful.
"""

prompt = PromptTemplate(
    template=template,
    input_variables=["topic"]
)

# Format the prompt with a specific topic
formatted_prompt = prompt.format(topic="supply and demand")

print("Formatted Prompt:")
print(formatted_prompt)

### 3b. Using the Template with the Model

Now let's use this template to generate responses for different topics.

In [ ]:
# Try different topics using the same template
topics = ["comparative advantage", "opportunity cost"]

for topic in topics:
    print(f"\n{'='*60}")
    print(f"Topic: {topic}")
    print('='*60)
    
    # Format the prompt
    formatted_prompt = prompt.format(topic=topic)
    
    # Get response from the model
    response = llm.invoke(formatted_prompt)
    
    print(response)

## 4. Chains: Combining Components Together

**Chains** are one of LangChain's core concepts. A chain combines a prompt template with a language model (and optionally other components) into a single, reusable pipeline.

### Why Use Chains?

Chains make your code:
- **Cleaner**: One call instead of format-then-invoke
- **Reusable**: Encapsulate the entire workflow
- **Composable**: Chains can be combined into more complex chains

The most basic chain is an **LLMChain** which combines a prompt template with a language model.

In [ ]:
from langchain.chains import LLMChain

# Create a chain that combines our prompt template with the model
chain = LLMChain(
    llm=llm,
    prompt=prompt
)

# Now we can use the chain with a single call
response = chain.run(topic="marginal utility")

print("Response from chain:")
print(response)

### 4a. A More Complex Template

Let's create a more sophisticated template that takes multiple variables.

In [ ]:
# Template with multiple variables
teaching_template = """
You are a {role} teaching a {course} course.

A student asks: "{question}"

Provide a clear, helpful answer appropriate for a {level} level student.
"""

teaching_prompt = PromptTemplate(
    template=teaching_template,
    input_variables=["role", "course", "question", "level"]
)

# Create a chain with this prompt
teaching_chain = LLMChain(
    llm=llm,
    prompt=teaching_prompt
)

# Use the chain
response = teaching_chain.run(
    role="economics professor",
    course="introductory microeconomics",
    question="Why do prices go up when demand increases?",
    level="freshman"
)

print(response)

## 5. Memory: Adding Conversation Context

So far, each call to the model has been independent. But what if we want the model to remember previous exchanges? That's where **Memory** comes in.

LangChain provides several memory types:
- `ConversationBufferMemory`: Stores the entire conversation history
- `ConversationBufferWindowMemory`: Keeps only the last N exchanges
- `ConversationSummaryMemory`: Stores a summary of the conversation

We'll use `ConversationBufferMemory` to build a simple chatbot.

### 5a. Setting up Memory

We need to:
1. Create a memory object
2. Create a prompt that includes the conversation history
3. Create a chain that uses both the prompt and memory

In [ ]:
from langchain.memory import ConversationBufferMemory
from langchain.chains import ConversationChain

# Create a memory object
memory = ConversationBufferMemory()

# Create a conversation chain (this automatically uses a conversation-friendly prompt)
conversation = ConversationChain(
    llm=llm,
    memory=memory,
    verbose=False  # Set to True to see the full prompt with history
)

print("✓ Conversation chain with memory created!")

### 5b. Having a Conversation

Now let's have a multi-turn conversation. Notice how the model remembers context from previous exchanges!

In [ ]:
# First message
response1 = conversation.predict(input="Hi! My name is Alex and I'm learning about economics.")
print("User: Hi! My name is Alex and I'm learning about economics.")
print(f"Assistant: {response1}\n")

In [ ]:
# Second message - references the first
response2 = conversation.predict(input="Can you explain what inflation is?")
print("User: Can you explain what inflation is?")
print(f"Assistant: {response2}\n")

In [ ]:
# Third message - the model should remember both previous exchanges
response3 = conversation.predict(input="What's my name again?")
print("User: What's my name again?")
print(f"Assistant: {response3}\n")

### 5c. Viewing the Conversation History

We can inspect what's stored in memory to see how LangChain tracks the conversation.

In [ ]:
# View the conversation history
print("Conversation History:")
print("="*60)
print(memory.buffer)
print("="*60)

## 6. Practical Example: Building a Study Buddy Chatbot

Let's combine everything we've learned to build a practical application: a study buddy chatbot that helps students learn economics concepts.

This chatbot will:
- Use a custom prompt template
- Remember the conversation
- Provide clear, student-friendly explanations

In [ ]:
from langchain.prompts import PromptTemplate

# Create a custom prompt for our study buddy
study_buddy_template = """
You are a friendly economics study buddy helping a college freshman prepare for exams.

Your guidelines:
- Be encouraging and supportive
- Explain concepts clearly with real-world examples
- If the student seems confused, break things down further
- Keep responses concise but informative

Previous conversation:
{history}

Student: {input}
Study Buddy:"""

study_prompt = PromptTemplate(
    input_variables=["history", "input"],
    template=study_buddy_template
)

# Create a new memory and conversation chain
study_memory = ConversationBufferMemory()

study_buddy = ConversationChain(
    llm=llm,
    memory=study_memory,
    prompt=study_prompt,
    verbose=False
)

print("✓ Study Buddy chatbot ready!")

### 6a. Testing the Study Buddy

Let's simulate a study session with multiple questions.

In [ ]:
# Study session
questions = [
    "I have an exam tomorrow on supply and demand. Can you help?",
    "What happens to price when supply decreases?",
    "Can you give me a real-world example?"
]

for question in questions:
    print(f"\n{'='*60}")
    print(f"Student: {question}")
    print('='*60)
    
    response = study_buddy.predict(input=question)
    print(f"Study Buddy: {response}")

## Summary

In this notebook, you learned:

1. **What LangChain is** and why it's useful for building LLM applications
2. **Prompt Templates** for creating reusable, parameterized prompts
3. **Chains** for combining prompts and models into workflows
4. **Memory** for building chatbots that remember conversation context
5. **Practical applications** like a study buddy chatbot

### Key Advantages of LangChain:
- **Abstraction**: Simplifies common patterns in LLM applications
- **Modularity**: Easy to swap components (different models, prompts, memory types)
- **Reusability**: Write once, use many times
- **Rich Ecosystem**: Many integrations with tools, databases, and APIs

### Comparison: With and Without LangChain

**Without LangChain** (from LlamaCpp notebook):
```python
messages = [{"role": "user", "content": f"Explain {topic}"}]
response = model.create_chat_completion(messages=messages)
```

**With LangChain** (this notebook):
```python
chain = LLMChain(llm=llm, prompt=template)
response = chain.run(topic=topic)
```

Both work, but LangChain shines when building more complex applications!

### Next Steps:
- Explore other LangChain features (agents, tools, vector databases)
- Try different memory types (window memory, summary memory)
- Build more complex chains (sequential chains, router chains)
- Integrate with external data sources